# Imports

In [ ]:
import torch
import gpytorch
from gpytorch.kernels import MaternKernel, ScaleKernel, AdditiveKernel
from gpytorch.constraints import Interval
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.mlls import ExactMarginalLogLikelihood
import numpy as np

dtype = torch.float32
device = 'cuda:0'

# Load Data

In [ ]:
# Data: N x 4 tensor [x, y, z, t], targets r in [0,1]
y_train = np.load('../data/y_train.npy')
n = y_train.shape[0]

n_samples = 1000
idx = np.random.choice(range(n), size=n_samples, replace=False)

X = torch.tensor(np.load('../data/X_train.npy')[idx], dtype=dtype, device=device)
r = torch.tensor(y_train[idx], dtype=dtype, device=device)
r_mean = r.mean()
r_std = r.std()

r = (r - r_mean) / r_std

# Precompute group means for mean function
mean_0 = r[X[:, 2] == 0].mean().item()
mean_1 = r[X[:, 2] == 1].mean().item()

# Define GP Model

In [ ]:
class GroupMean(gpytorch.means.Mean):
    def __init__(self, mean_0, mean_1):
        super().__init__()
        self.register_parameter("mean_0", torch.nn.Parameter(torch.tensor(mean_0)))
        self.register_parameter("mean_1", torch.nn.Parameter(torch.tensor(mean_1)))

    def forward(self, x):
        z = x[:, 2]  # indoor/outdoor indicator
        return torch.where(z == 1, self.mean_1.expand(x.shape[0]),
                                   self.mean_0.expand(x.shape[0]))


class HeteroscedasticGP(gpytorch.models.ExactGP):
    def __init__(self, X, r, likelihood, mean_0, mean_1):
        super().__init__(X, r, likelihood)
        
        self.mean_module = GroupMean(mean_0, mean_1)

        # K_xyz: acts on dims [0,1,2] (x, y, z)
        self.covar_xy = ScaleKernel(
            MaternKernel(nu=2.5, ard_num_dims=2, active_dims=(0, 1))
        )
        self.covar_z = ScaleKernel(
            MaternKernel(nu=2.5, ard_num_dims=1, active_dims=(2,),
                         lengthscale_constraint=Interval(1e-3, 0.5))
        )
        self.covar_t = ScaleKernel(
            MaternKernel(nu=2.5, ard_num_dims=1, active_dims=(3,),
                         lengthscale_constraint=Interval(1e-3, 0.2))
        )

    def forward(self, x):
        mean = self.mean_module(x)
        covar = self.covar_xy(x) + self.covar_z(x) + self.covar_t(x)
        return gpytorch.distributions.MultivariateNormal(mean, covar)

# Build per-observation noise vector from group membership
def make_noise_vector(X_data, log_var_0, log_var_1):
    z = X_data[:, 2]
    noise = torch.where(z == 1, log_var_1.exp(), log_var_0.exp())
    return noise

class HeteroscedasticLikelihood(gpytorch.likelihoods.FixedNoiseGaussianLikelihood):
    """Wraps FixedNoise but exposes σ²_0, σ²_1 as learnable params."""
    pass

# MLL Optimization

In [ ]:
from torch.optim import Adam

def initialize_hyperparams(model, X, r):
    # Lengthscales: median pairwise distance heuristic
    xy = X[:, :2]
    z  = X[:, 2:3]
    t   = X[:, 3:4]
    
    with torch.no_grad():
        ls_xy = torch.cdist(xy, xy).median().clamp(min=1e-3)
        ls_z = torch.cdist(z, z).median().clamp(min=1e-3, max=0.5)
        ls_t   = torch.cdist(t, t).median().clamp(min=1e-3, max=0.2)

        model.covar_xy.base_kernel.lengthscale = ls_xy
        model.covar_xy.outputscale = r.var()
        model.covar_z.base_kernel.lengthscale = ls_z
        model.covar_z.outputscale = r.var() / 10
        model.covar_t.base_kernel.lengthscale = ls_t
        model.covar_t.outputscale = r.var() / 10

def fit_marginal_likelihood(X, r, n_iter=500):
    noise_vec = make_noise_vector(X, log_var_0, log_var_1)
    likelihood = gpytorch.likelihoods.FixedNoiseGaussianLikelihood(
        noise=noise_vec, learn_additional_noise=False
    )
    model = HeteroscedasticGP(X, r, likelihood, mean_0, mean_1).to(device)
    initialize_hyperparams(model, X, r)

    # All learnable params: kernel hyperparams + log_var_0, log_var_1
    params = list(model.parameters()) + [log_var_0, log_var_1]
    optimizer = Adam(params, lr=0.1)
    mll = ExactMarginalLogLikelihood(likelihood, model)

    model.train(); likelihood.train()
    for i in range(n_iter):
        optimizer.zero_grad()
        # Recompute noise vector at current σ² values
        noise_vec = make_noise_vector(X, log_var_0, log_var_1)
        likelihood.noise = noise_vec
        with gpytorch.settings.fast_pred_var(), \
         gpytorch.settings.cg_tolerance(1e-3), \
         gpytorch.settings.max_cg_iterations(100), \
         gpytorch.settings.num_trace_samples(10):   # stochastic trace for MLL
            output = model(X)
            loss = -mll(output, r)
        loss.backward()
        optimizer.step()
        if i % 50 == 0:
            print(f"[{i}] MLL: {-loss.item():.4f} | "
                  f"σ²_0={log_var_0.exp():.4f} σ²_1={log_var_1.exp():.4f}")

    return model, likelihood

In [ ]:
# Initialize with FixedNoiseGaussianLikelihood; update noise each forward pass
noise_init = (r.var() / 2).log().item()
log_var_0 = torch.nn.Parameter(torch.tensor(noise_init, device=device))  # ~0.13 initial noise
log_var_1 = torch.nn.Parameter(torch.tensor(noise_init, device=device))

model, likelihood = fit_marginal_likelihood(X, r, n_iter=501)

In [ ]:
model.eval()
with torch.no_grad(), gpytorch.settings.fast_pred_var(False):
    K_fixed_gpu = model.covar_xy(X).evaluate() + model.covar_z(X).evaluate() + model.covar_t(X).evaluate()

print("K_xy lengthscales:", model.covar_xy.base_kernel.lengthscale.detach().cpu())
print("K_xy outputscale: ", model.covar_xy.outputscale.detach().cpu())
print("K_z lengthscales:", model.covar_z.base_kernel.lengthscale.detach().cpu())
print("K_z outputscale: ", model.covar_z.outputscale.detach().cpu())
print("K_t  lengthscale: ", model.covar_t.base_kernel.lengthscale.detach().cpu())
print("K_t  outputscale: ", model.covar_t.outputscale.detach().cpu())

# Posterior Inference of Variances

In [ ]:
import numpyro
import numpyro.distributions as dist
from numpyro.infer import NUTS, MCMC
import jax.numpy as jnp

def noise_model(K_fixed_jax, r_jax, z_jax, group_means_jax):
    sigma2_0 = numpyro.sample("sigma2_0", dist.InverseGamma(1e-3, 1e-3))
    sigma2_1 = numpyro.sample("sigma2_1", dist.InverseGamma(1e-3, 1e-3))

    sigma2_vec = jnp.where(z_jax == 1, sigma2_1, sigma2_0)
    K_noisy = K_fixed_jax + jnp.diag(sigma2_vec)

    numpyro.sample("r_obs",
                   dist.MultivariateNormal(group_means_jax, K_noisy),
                   obs=r_jax)

# Convert tensors from torch → jax
import jax
K_fixed_jax = jax.dlpack.from_dlpack(K_fixed_gpu)
r_jax = jax.dlpack.from_dlpack(r.contiguous())
z_jax = jnp.array(X[:, 2].cpu().numpy()).astype(jnp.int32)
mu_jax = jnp.where(X[:, 2].cpu().numpy() == 1, mean_1, mean_0)

kernel = NUTS(noise_model)
# mcmc   = MCMC(kernel, num_warmup=10, num_samples=100, num_chains=1, chain_method='vectorized')
# mcmc.run(jax.random.PRNGKey(0), K_fixed_jax, r_jax, z_jax, mu_jax)

# Prediction on Grid

In [ ]:
def predict(model, likelihood, X_star, temporal=True):
    """
    temporal=False: use only K_xy + K_z (drop K_t) for out-of-sample time.
    """
    model.eval(); likelihood.eval()
    
    if not temporal:
        # Monkey-patch: zero out K_t contribution for predictions
        original_forward = model.forward
        def forward_no_t(x):
            mean = model.mean_module(x)
            covar = model.covar_xy(x) + model.covar_z(x)  # K_xyz only
            return gpytorch.distributions.MultivariateNormal(mean, covar)
        model.forward = forward_no_t

    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        noise_star = torch.where(X_star[:,2].bool(), log_var_0.exp().item(), log_var_0.exp().item())
        pred = likelihood(model(X_star), noise=noise_star)
        mean = pred.mean
        lower, upper = pred.confidence_region()  # ±2σ

    if not temporal:
        model.forward = original_forward
    
    return mean, lower, upper

# Build prediction grid
X_star = torch.tensor(np.load('../data/X_test.npy'), dtype=dtype, device=device)

mean_pred, lo, hi = predict(model, likelihood, X_star, temporal=False)

In [ ]:
r_hat = mean_pred * r_std + r_mean
r_orig = r * r_std + r_mean

# Plot Results

In [ ]:
import sys
import os
sys.path.append(os.getcwd())
sys.path.append(f"{os.getcwd()}/..")


from wifiplotting import *

import dill

with open('../data/osm_context.pkl', 'rb') as f:
    osm_context = dill.load(f)
wlon_train, wlat_train = np.load('../data/world_train.npy').T
wlon_test, wlat_test = np.load('../data/world_test.npy').T

In [ ]:
with torch.no_grad():
    vmax = torch.quantile(r_orig, 0.975)
    vmin = torch.quantile(r_orig, 0.025)
    
    base_fig, base_ax, osm_metadata = osm_context.generate_base_axis(draw_buildings=False)
    
    sc = base_ax.scatter(wlon_test, wlat_test, c=y_hat.cpu(), cmap="RdYlGn", vmin=vmin, vmax=vmax)
    
    # tr = base_ax.scatter(wlon_train[idx], wlat_train[idx], c=(r*r_std+r_mean).cpu(), cmap="RdYlGn",
    #                 vmin=vmin, vmax=vmax, alpha=1)
    plt.ticklabel_format(style='plain', axis='both', useOffset=False)
    
    plt.colorbar(sc)
    plt.tight_layout()